# 多层感知机的从零开始实现

##### 设定批量大小（该数值越大，训练更快显存也更大）
##### 调用d2l函数：
1. 自动下载（若本地无缓存）并读取 Fashion-MNIST 数据集。
2. 进行标准化和张量转换。resize and trans
3. 返回两个 PyTorch DataLoader ： train_iter （训练集）和 test_iter （测试集），每次迭代产生一个大小为 batch_size 的小批量。

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

#### 初始化模型参数 ：w1 ，b1 ，w2 ，b2
权重和偏置最初随机初始化，通过反向传播和梯度下降在训练中自动学习。

隐藏层权重w1，b1 输出层权重w2，b2
* W1, b1 定义从输入到隐藏层的参数
* W1 :矩阵形状 [784, 256] ，把展平后的输入 784 映射到隐藏层 256 。
* b1 : 矩阵形状[256] ，隐藏层偏置。
* W2 : 矩阵形状[256, 10] ，把隐藏表示映射到 10 类的输出。
* b2 : 矩阵形状[10] ，输出层偏置。


##### 隐藏层取256是经验常用值，常使用为2的若干次幂，可改为 128 、 512 等；过小易欠拟合、过大易过拟合且训练更慢。
前向传播通常是 H = ReLU(X @ W1 + b1) ，随后 logits = （H @ W2 + b2）完成分类，x指的是输入的图像经展平后

In [ ]:
#输入层：28×28=784（把每张图展平为长度784的向量作为MLP的输入）
#10是输出维度：数据集有 10 个类别，模型最终需要输出 10 维的分类得分（logits），对应 0-9 的类别索引
#256是隐藏层宽度：两层感知机的中间层神经元数量，属于可调的超参数，用来控制模型容量与训练开销的平衡。
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

##### 采用激活函数：Relu

In [ ]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

#### 定义我们的模型net
##### 其中使用reshape将每个二维图像转换为一个长度为num_inputs的向量

In [ ]:
def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X@W1 + b1)  # 这里“@”代表矩阵乘法
    return (H@W2 + b2)

##### 定义损失函数

In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')

#### 训练
- net ：前向函数/模型，输入一批图像张量，输出每类的打分（ [batch, 10] 的 logits）。
- train_iter ：训练集 DataLoader ，按批提供 (X, y) 。
- test_iter ：测试集 DataLoader ，用于周期性评估准确率。
- loss ：损失函数，这里是 nn.CrossEntropyLoss(reduction='none') ，对每个样本分别给出损失。
- num_epochs ：训练轮数，这里是 10 。
- updater ：参数更新器，这里是 torch.optim.SGD(params, lr=0.1) 。

#### train_ch3 做什么
- 训练循环：运行 num_epochs 次，每次遍历 train_iter 的所有批次。
- 批次内步骤：
  - 前向： y_hat = net(X)
  - 损失： l = loss(y_hat, y) ；若 updater 是 PyTorch 优化器，会用 l.mean().backward() 求梯度。
  - 更新： updater.zero_grad() 、 backward() 、 updater.step() 。
  - 统计：累计训练损失和训练准确率（用 d2l.accuracy 计算）。
- 验证评估：每个 epoch 后，用 test_iter 计算测试准确率（推理阶段不求梯度）。
- 可视化与输出：绘制或打印每轮的训练损失、训练准确率、测试准确率，便于观察收敛情况。

In [ ]:
num_epochs, lr = 10, 0.1
updater = torch.optim.SGD(params, lr=lr)
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, updater)

验证

In [ ]:
d2l.predict_ch3(net, test_iter)